To study:
 	
Efficient numerical simulation of complex Josephson quantum circuits, Andrew J. Kerman (https://doi.org/10.48550/arXiv.2010.14929)


# C-shunted qubit

<img src="cshunttext.jpeg" alt="Circuit" width="300" height="300"/>

In this Notebook, the spectrum of a c-shunted qubit with an applied external flux will be analyzed. The circuit can be seen above. The red markings denote the nodes, the red numbers the node number.

First we want to plot the excitation profile of the circuit with an external flux applied, later we will run an optimization of the parameters (values of the components $J_i$, $C_i$, $L$ with $i=1,2,3$) based on target data for the excitation profile, which can be found in 'target_fluxqubit.p'.

## Basics

First we need to import the necessary things. The different representations (Charge, Flux, Kerman, Oscillator) are imported from circuit.py. The circuit implementation is found in models.py, the optimization with Gradient Descent in optimization, loss ... Further on a good amount of handsy plot tools can be found in utils.py.

In [3]:
from des_scq import models,circuit
from des_scq.circuit import Kerman,Charge
from des_scq.optimization import Optimization
from des_scq.discovery import lossTransition
from des_scq.components import capE,indE,e,h
from torch import tensor, float32 as float, stack, float64 as double
from numpy import arange,linspace,array
from des_scq.utils import plotCompare,plotHeatmap,plotOptimization
import pickle
from torch import set_num_threads
set_num_threads(32)

#### Conversion of units
From components, the functions capE,indE,e,h are imported. These convert physical units for capacitors (Farad) and inductors (Henry) into energy units (GHz), which is done for the capacitors $C_i$ in the cell below.

In [4]:
C1 = capE(45e-15)
C2 = capE(1e-15)
C3 = C1

print('The capacity of C1 and C3 is', C1, 'GHz')
print('The capacity of C2 is', C2, 'GHz')

The capacity of C1 and C3 is 0.4304495401712799 GHz
The capacity of C2 is 19.370229307707593 GHz


#### Choose basis and circuit
In the next step we chose which basis/representation we want to work in and the size of it. After that we can define the circuit with the shuntedQubit() function from models.py corresponding to the given circuit and the defined values for $C_i$.

In [7]:
rep = Charge #choose Charge basis
basis = [8,8,8] #choose size of Charge basis

#shuntedQubit() takes representation, basis, capacitor values and sparse option as argument
#read more about sparse in 'sparse_or_dense?.txt' located in 'additional information' folder 

circuit = models.shuntedQubit(rep,basis,cap=[C1,C2,C3])
print(circuit.circuitComponents())

H_LC = circuit.hamiltonianLC() #was circuit.chargeHamiltonianLC
H_J = circuit.hamiltonianJosephson #was circuit.josephsonCharge

{'JJ1': 119.99999999999999, 'C1': 1.1615763366852379, 'JJ2': 50.000000000000014, 'C2': 0.025812807481894157, 'JJ3': 119.99999999999999, 'C3': 1.1615763366852379, 'I': 0.00025330295910584456}


Next we want to work in Kerman basis:

In [9]:
rep = Kerman #choose Kerman basis
basis = {'O':[4],'J':[8,8],'I':[]} #choose size of Kerman basis
circuit = models.shuntedQubit(rep,basis,cap=[C1,C2,C3])

/Users/chishti/des-scq/.venv/lib/python3.13/site-packages/des_scq/circuit.py:399: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Copy.cpp:308.)
  R = diagonalisation(Ln_.detach(),True).to(float)


In Kerman basis we can now analyze the distribution of modes with kermanDistribution(). Here we can determine the number of oscillator, island, and josephson modes.

The number of oscillator modes $N_o$ is equal to the rank of the inverse node inductance matrix $L_n^{-1}$, so the number of eigenvectors of $L_n$ with non-zero eigenvalues.
The number of island modes $N_i$ corresponds to the number of modes, whose fluxiod coordinates do not appear in the Hamiltonian.
The number of Josephson modes $N_j$ then equals to $N_j = N_n -N_i-N_o$ where $N_n$ denotes the total number of nodes.

For more detail read the paper, that is referenced in cell 1 and the implementation of IslandModes() in circuit.py.

In [10]:
nodeDistribution = circuit.kermanDistribution()
print('The number of oscillator modes N_o is',nodeDistribution[0])
print('The number of island modes N_i is',nodeDistribution[1])
print('The number of josephson modes N_j is',nodeDistribution[2])

The number of oscillator modes N_o is 1
The number of island modes N_i is 0
The number of josephson modes N_j is 2


#### Flux
Now we want to define an external flux. We chose the a range from 0, which corresponds to no flux, to 1, which corresponds zu one flux quantum $\phi_0$. The flux profile must be a list of tensors, otherwise further computations won't work.

In [11]:
flux_range = tensor(linspace(0,1,5)) #need tensor for optimization
flux_profile = [[flux] for flux in flux_range] #list of tensors

## Pre-optimization profile

We now want to look at the excitation profile. For this the Hamiltonian of the circuit needs to be determined

In [12]:
H_LC = circuit.hamiltonianLC() #compute LC hamiltonian in kerman basis
H_J = circuit.hamiltonianJosephson #compute josephson hamiltonian in kerman basis

Next we want to compute the spectrum (the energies $E_i$ for $i=0,1,2,3$ and the eigenstates) for each flux point of flux_profile.

In [13]:
preoptim_Spectrum = circuit.spectrumManifold(flux_profile)

We are only interested in the eigenenergies:

In [14]:
pre_Ex = [spectrum for spectrum,state in preoptim_Spectrum] #extract eigenenergies
pre_Ex = stack(pre_Ex).T #transpose so all E0 are in one list, E1 in another...
pre_Ex = pre_Ex.detach().numpy() #dump the tensor
pre_Ex = pre_Ex - pre_Ex[0] #norm energies on ground state

print('The transition energies E(1->0) for the different flux values in flux_range are', pre_Ex[1])
print('The transition energies E(2->1) for the different flux values in flux_range are', pre_Ex[2]-pre_Ex[1])
print('The transition energies E(2->0) for the different flux values in flux_range are', pre_Ex[2])

The transition energies E(1->0) for the different flux values in flux_range are [16.03485264 20.95520046 10.32645091 20.95520048 16.03485264]
The transition energies E(2->1) for the different flux values in flux_range are [ 5.01742849  7.31665525 12.24363212  7.31665486  5.01742849]
The transition energies E(2->0) for the different flux values in flux_range are [21.05228114 28.27185571 22.57008303 28.27185534 21.05228114]


Now visualize excitation profile pre-optimization including the anharmonicity $A$ defined as 
$A=E_{20} - 2 E_{10}$.
With plotCompare found in utils.py we have a convenient way of visualizing the profile while being able dynamically change the scale.

In [15]:
plotCompare(flux_range,{'E10':pre_Ex[1],'E21':pre_Ex[2]-pre_Ex[1],'anharmonicity':pre_Ex[2]-2*pre_Ex[1]},
            'Excitation Profile : pre-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

## Optimization

Now the spectrum will be optimized based on the target data. The file for the target data can be found in the same folder as this notebook an we import it by:

In [16]:
with open('target_fluxqubit.pick', 'rb') as content: target_info = pickle.load(content)
target_spectrum = target_info['spectrum']
E10 = target_spectrum[:,1] - target_spectrum[:,0] #get energies for E(1->0)
E21 = target_spectrum[:,2] - target_spectrum[:,1] #get energies for E(2->1)

Put relevant data from $E10$, $E21$ in dictionary. We choose 5 equally spaced points, just as in fluxprofile.

In [17]:
target = {'E10':E10[[0,10,20,30,-1]],'E21':E21[[0,10,20,30,-1]]}

Now optimize the spectrum. The function lossObjective will give the loss function for the optimization with SGD. lossTransition is a function found in discovery.py, that computes the loss function for the energie transitions. The loss is based on MSE.

For more detail look at the definition of loss Objective.

The optimizer will be GradientDescent which takes circuit, circuit profile, flux profile, and loss function as an argument.

In [19]:
lossObjective = lossTransition(tensor(target['E10'],dtype=double),tensor(target['E21'],dtype=double)) #define loss function
optim = Optimization(circuit,flux_profile,lossObjective) #choose Gradien Descent as optimizer
optim.initAlgo(lr=.5e-1) # set learning rate
print(optim.optimizer)
dLogs,dParams,dCircuit = optim.optimization(iterations=10) #run optimization, set number of iterations

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    weight_decay: 0
)


In [20]:
plotCompare(dLogs.index,dLogs[['loss']],'Optimization Flux Profile C - Shunted Qubit','iteration',export='pdf',size=(600,800)) 
# plot loss vs. iterations

In [21]:
plotCompare(dCircuit.index,dCircuit,'Optimization-Circuit Parameters','iteration','energy(GHz)',export='pdf',size=(600,800))
# plot circuit parameter energy value vs. iterations

## Post-optimization profile
Now plot the post optimization profile. All the steps here are analogous to the steps in section "Pre-optimization profile"

In [22]:
H_LC = circuit.hamiltonianLC() #compute updated LC hamiltonian in kerman basis
H_J = circuit.hamiltonianJosephson #compute updated josephson hamiltonian in kerman basis

In [23]:
postoptim_Spectrum = circuit.spectrumManifold(flux_profile)

In [24]:
post_Ex = [spectrum for spectrum,state in postoptim_Spectrum] #extract eigenenergies
post_Ex = stack(post_Ex).T #transpose so all E0 are in one list, E1 in another...
post_Ex = post_Ex.detach().numpy() #dump the tensor
post_Ex = post_Ex - post_Ex[0] #norm energies on ground state

In [25]:
plotCompare(flux_range,{'E10':post_Ex[1],'E21':post_Ex[2]-post_Ex[1],'anharmonicity':post_Ex[2]-2*post_Ex[1]},
            'Excitation Profile : post-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

Compare excitation profile pre and post optimization.

In [26]:
plot = {'post':post_Ex[1],'pre':pre_Ex[1],'Target':target['E10']}
plotCompare(flux_range,plot,
            'Excitation(E10) Profile','external flux','energy(GHz)',export='pdf',size=(600,800))

In [27]:
plot = {'post IInd':post_Ex[2],'pre IInd':pre_Ex[2],'Target21':target['E21']}
plotCompare(flux_range,plot,
            'Excitation(E21) Profile','external flux','energy(GHz)',export='pdf',size=(600,800))

## Loss Scape

In [28]:
print(circuit.circuitComponents())
static = {'C1': 0.478,
          'JJ2': 112.336, 'C2': 0.034,
           'C3': 1.676,
          'I': 0.000221}

{'JJ1': 109.61755802584022, 'C1': 0.7151255569180753, 'JJ2': 74.8947095935569, 'C2': 0.021227919059411265, 'JJ3': 150.5486611314448, 'C3': 1.52479501012469, 'I': 0.0002810674611443518}


In [29]:
JJ1 = linspace(100,150,5,endpoint=True)
JJ3 = linspace(100,700,5,endpoint=True)
scape = {'JJ1':JJ1,'JJ3':JJ3}

In [31]:
Loss = optim.lossScape(scape,static)

In [32]:
path = {'Adam':dCircuit[['JJ1','JJ3']].to_numpy()}
plotOptimization(Loss,JJ1,JJ3,path,'Optimization-Flux profiling','JJ1(GHz)','JJ3(GHz)',export='pdf',size=(600,800))